In [ ]:
# 前面部分和上节课一样

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [3]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [4]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [ ]:
#开启变化

In [5]:
#稍后在比较手动梯度和 PyTorch 梯度时将使用的实用函数
def cmp(s, dt, t):
  ex = torch.all(dt == t.grad).item()
  app = torch.allclose(dt, t.grad)
  maxdiff = (dt - t.grad).abs().max().item()
  print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [6]:
n_embd = 10 # 字符嵌入向量的维度
n_hidden = 64 # MLP 隐藏层的神经元数量

g = torch.Generator().manual_seed(2147483647) # 设置随机种子以确保结果可复现

C  = torch.randn((vocab_size, n_embd), generator=g)

# 第 1 层
# 使用了 Kaiming 初始化增益 (5/3)，适用于 tanh 激活函数
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden, generator=g) * 0.1 # 这里加上 b1 只是为了好玩，由于 BatchNorm (BN) 的存在，它实际上是不起作用的

# 第 2 层
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0.1

# Batch Normalization (批归一化) 参数
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

# 注意：我使用了非标准的方式来初始化许多参数。
# 这是因为有时如果全部初始化为 0，可能会掩盖反向传播（backward pass）实现中的错误。

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # 打印参数总数

for p in parameters:
  p.requires_grad = True

4137


In [7]:
batch_size = 32
n = batch_size # 为了方便，定义一个更短的变量名 n

# 构建一个小批量数据 (minibatch)
# 在训练集 Xtr 的行数范围内，随机生成 batch_size 个索引
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)

# 根据随机索引提取对应的输入 X 和标签 Y
Xb, Yb = Xtr[ix], Ytr[ix] # 获取当前的批处理数据 X 和 Y

In [8]:
# 前向传播，将其“拆解”为更小的步骤，以便我们可以逐一进行反向传播
emb = C[Xb] # 将字符嵌入为向量
embcat = emb.view(emb.shape[0], -1) # 将这些向量拼接起来

# 线性层 1
hprebn = embcat @ W1 + b1 # 隐藏层预激活值

# BatchNorm 层 (批归一化)
bnmeani = 1/n*hprebn.sum(0, keepdim=True) # 计算均值
bndiff = hprebn - bnmeani # 减去均值
bndiff2 = bndiff**2 # 平方偏差
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # 方差：注意这里使用了贝塞尔修正（除以 n-1 而不是 n）
bnvar_inv = (bnvar + 1e-5)**-0.5 # 方差的平方根倒数（标准差的倒数）
bnraw = bndiff * bnvar_inv # 归一化后的原始值
hpreact = bngain * bnraw + bnbias # 应用缩放（gain）和偏移（bias）

# 非线性层
h = torch.tanh(hpreact) # 隐藏层激活值

# 线性层 2
logits = h @ W2 + b2 # 输出层（未归一化的对数概率）

# 交叉熵损失函数（等同于 F.cross_entropy(logits, Yb)）
logit_maxes = logits.max(1, keepdim=True).values # 每一行的最大值
norm_logits = logits - logit_maxes # 减去最大值，确保数值稳定性（防止 exp 爆炸）
counts = norm_logits.exp() # 取指数
counts_sum = counts.sum(1, keepdims=True) # 对指数求和
counts_sum_inv = counts_sum**-1 # 求和的倒数。注意：如果用 (1.0 / counts_sum) 可能导致反向传播时的精度无法完全一致
probs = counts * counts_sum_inv # 得到概率分布
logprobs = probs.log() # 取对数
loss = -logprobs[range(n), Yb].mean() # 计算负对数似然损失

# PyTorch 反向传播（用于对比验证）
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, 
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
          bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
          embcat, emb]:
  t.retain_grad() # 显式保留中间变量的梯度，以便后续检查
loss.backward()
loss

tensor(3.3300, grad_fn=<NegBackward0>)

In [ ]:
#练习 1：手动反向传播整个过程
# 逐个反向传播所有变量
# 按照上面前向传播中定义的顺序，逐个反向传播

# -----------------
#请输入代码开始练习，每完成一个，把下方的cmp前的#去掉，运行查看是否正确
# -----------------
# cmp('logprobs', dlogprobs, logprobs)
# cmp('probs', dprobs, probs)
# cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
# cmp('counts_sum', dcounts_sum, counts_sum)
# cmp('counts', dcounts, counts)
# cmp('norm_logits', dnorm_logits, norm_logits)
# cmp('logit_maxes', dlogit_maxes, logit_maxes)
# cmp('logits', dlogits, logits)
# cmp('h', dh, h)
# cmp('W2', dW2, W2)
# cmp('b2', db2, b2)
# cmp('hpreact', dhpreact, hpreact)
# cmp('bngain', dbngain, bngain)
# cmp('bnbias', dbnbias, bnbias)
# cmp('bnraw', dbnraw, bnraw)
# cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
# cmp('bnvar', dbnvar, bnvar)
# cmp('bndiff2', dbndiff2, bndiff2)
# cmp('bndiff', dbndiff, bndiff)
# cmp('bnmeani', dbnmeani, bnmeani)
# cmp('hprebn', dhprebn, hprebn)
# cmp('embcat', dembcat, embcat)
# cmp('W1', dW1, W1)
# cmp('b1', db1, b1)
# cmp('emb', demb, emb)
# cmp('C', dC, C)

In [9]:
#练习1正确答案

dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0/n
dprobs = (1.0 / probs) * dlogprobs
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
dcounts = counts_sum_inv * dprobs
dcounts_sum = (-counts_sum**-2) * dcounts_sum_inv
dcounts += torch.ones_like(counts) * dcounts_sum
dnorm_logits = counts * dcounts
dlogits = dnorm_logits.clone()
dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True)
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes
dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)
dhpreact = (1.0 - h**2) * dh
dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
dbnraw = bngain * dhpreact
dbnbias = dhpreact.sum(0, keepdim=True)
dbndiff = bnvar_inv * dbnraw
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv
dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar
dbndiff += (2*bndiff) * dbndiff2
dhprebn = dbndiff.clone()
dbnmeani = (-dbndiff).sum(0)
dhprebn += 1.0/n * (torch.ones_like(hprebn) * dbnmeani)
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)
demb = dembcat.view(emb.shape)
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
  for j in range(Xb.shape[1]):
    ix = Xb[k,j]
    dC[ix] += demb[k,j]
    
cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
cmp('C', dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: True  | approximate: True  | maxdiff: 0.0
bngain          | exact: True  | approximate: True  | maxdiff: 0.0
bnbias          | exact: True  | approximate: True  | maxdiff: 0.0
bnraw           | exact: True  | approximate: True  | maxdiff:

In [10]:
# 练习 2：一次性完成交叉熵反向传播
# 要完成此挑战，请查看损失函数的数学表达式，
# 求导，简化表达式，然后写出结果。

# forward pass

# before:
# logit_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logit_maxes # subtract max for numerical stability
# counts = norm_logits.exp()
# counts_sum = counts.sum(1, keepdims=True)
# counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
# probs = counts * counts_sum_inv
# logprobs = probs.log()
# loss = -logprobs[range(n), Yb].mean()

# now:
loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())

3.3299856185913086 diff: -2.384185791015625e-07


In [ ]:
# backward pass

# -----------------
# YOUR CODE HERE :)
dlogits = None # 提示：三行代码
# -----------------

#cmp('logits', dlogits, logits) # 我只能得到近似值，我的最大差异是 6e-9

In [11]:
#练习2答案
# backward pass

dlogits = F.softmax(logits, 1)
dlogits[range(n), Yb] -= 1
dlogits /= n

cmp('logits', dlogits, logits)

logits          | exact: False | approximate: True  | maxdiff: 1.0477378964424133e-08


In [ ]:
# 练习 3：一次性完成批归一化反向传播

# 要完成此挑战，请查看批归一化输出的数学表达式，
# 对其输入求导，简化表达式，然后写出结果
# 批归一化论文：https://arxiv.org/abs/1502.03167

# forward pass

# before:
# bnmeani = 1/n*hprebn.sum(0, keepdim=True)
# bndiff = hprebn - bnmeani
# bndiff2 = bndiff**2
# bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
# bnvar_inv = (bnvar + 1e-5)**-0.5
# bnraw = bndiff * bnvar_inv
# hpreact = bngain * bnraw + bnbias

# now:
hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5) + bnbias
print('max diff:', (hpreact_fast - hpreact).abs().max())

In [ ]:
# backward pass

# before we had:
# dbnraw = bngain * dhpreact
# dbndiff = bnvar_inv * dbnraw
# dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
# dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv
# dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar
# dbndiff += (2*bndiff) * dbndiff2
# dhprebn = dbndiff.clone()
# dbnmeani = (-dbndiff).sum(0)
# dhprebn += 1.0/n * (torch.ones_like(hprebn) * dbnmeani)

# 根据 dhpreact 计算 dhprebn（即通过批归一化进行反向传播）
#（您还需要使用上面前向传播中的一些变量）

# -----------------
# YOUR CODE HERE :)
dhprebn = None # TODO. my solution is 1 (long) line
# -----------------

cmp('hprebn', dhprebn, hprebn) # I can only get approximate to be true, my maxdiff is 9e-10

In [12]:
#练习3 答案
dlogits = F.softmax(logits, 1)
dlogits[range(n), Yb] -= 1
dlogits /= n

cmp('logits', dlogits, logits)#我只能得到近似值，我的最大差异是 6e-9

logits          | exact: False | approximate: True  | maxdiff: 1.0477378964424133e-08


In [ ]:
# 练习 4：大功告成！
# 使用你自己编写的反向传播来训练这个 MLP 神经网络

# 初始化
n_embd = 10 # 字符嵌入向量的维度
n_hidden = 200 # MLP 隐藏层的神经元数量

g = torch.Generator().manual_seed(2147483647) # 设置随机种子以确保结果可复现
C  = torch.randn((vocab_size, n_embd),            generator=g)
# 第 1 层
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1
# 第 2 层
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm 参数
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # 打印参数总数
for p in parameters:
  p.requires_grad = True

# 保持与上次相同的优化配置
max_steps = 200000
batch_size = 32
n = batch_size # 为了方便
lossi = []

# 一旦你写好了反向传播，为了效率请使用这个上下文管理器 (TODO)
#with torch.no_grad():

# 开启优化循环
for i in range(max_steps):

  # 构建小批量数据 (minibatch)
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
  Xb, Yb = Xtr[ix], Ytr[ix] # 批处理数据 X, Y

  # 前向传播
  emb = C[Xb] # 将字符嵌入为向量
  embcat = emb.view(emb.shape[0], -1) # 拼接向量
  # 线性层
  hprebn = embcat @ W1 + b1 # 隐藏层预激活
  # BatchNorm 层
  # -------------------------------------------------------------
  bnmean = hprebn.mean(0, keepdim=True)
  bnvar = hprebn.var(0, keepdim=True, unbiased=True)
  bnvar_inv = (bnvar + 1e-5)**-0.5
  bnraw = (hprebn - bnmean) * bnvar_inv
  hpreact = bngain * bnraw + bnbias
  # -------------------------------------------------------------
  # 非线性层
  h = torch.tanh(hpreact) # 隐藏层
  logits = h @ W2 + b2 # 输出层
  loss = F.cross_entropy(logits, Yb) # 损失函数

  # 反向传播
  for p in parameters:
    p.grad = None
  loss.backward() # 用于正确性对比，稍后删除！

  # 手动反向传播！
  # -----------------
  # 在此处编写你的代码 :)
  dC, dW1, db1, dW2, db2, dbngain, dbnbias = None, None, None, None, None, None, None
  grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]
  # -----------------

  # 更新参数
  lr = 0.1 if i < 100000 else 0.01 # 阶梯式学习率衰减
  for p, grad in zip(parameters, grads):
    p.data += -lr * p.grad # 旧方法：弱小狗(cheems doge)模式（使用 PyTorch 生成的 .grad）
    #p.data += -lr * grad # 新方法：肌肉狗模式 TODO: 准备好后启用

  # 追踪统计数据
  if i % 10000 == 0: # 隔一段时间打印一次
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())

  if i >= 100: # TODO: 当你准备好训练整个网络时，删除这个提前中断语句
    break

In [19]:
# 练习 4：大功告成！
# 使用你自己编写的反向传播来训练这个 MLP 神经网络

# 初始化参数
n_embd = 10 # 字符嵌入向量的维度
n_hidden = 200 # MLP 隐藏层的神经元数量

g = torch.Generator().manual_seed(2147483647) # 设置随机种子以确保结果可复现
C = torch.randn((vocab_size, n_embd), generator=g)

# 第 1 层
# 使用 Kaiming 初始化增益 (5/3)，适用于 tanh 激活函数
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden, generator=g) * 0.1

# 第 2 层
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0.1

# BatchNorm 参数
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # 打印参数总数

for p in parameters:
  p.requires_grad = True

# 优化设置
max_steps = 200000
batch_size = 32
n = batch_size # 为了方便
lossi = []

# 一旦你写好了反向传播，为了效率请使用这个上下文管理器
with torch.no_grad():

  # 开启优化循环
  for i in range(max_steps):

    # 构建小批量数据 (minibatch construct)
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix] # 获取当前的批处理数据 X 和 Y

    # 前向传播 (forward pass)
    emb = C[Xb] # 将字符嵌入为向量
    embcat = emb.view(emb.shape[0], -1) # 拼接向量
    
    # 线性层 1
    hprebn = embcat @ W1 + b1 # 隐藏层预激活值

    # BatchNorm 层
    # -------------------------------------------------------------
    bnmean = hprebn.mean(0, keepdim=True)
    bnvar = hprebn.var(0, keepdim=True, unbiased=True)
    bnvar_inv = (bnvar + 1e-5)**-0.5
    bnraw = (hprebn - bnmean) * bnvar_inv
    hpreact = bngain * bnraw + bnbias
    # -------------------------------------------------------------

    # 非线性激活 (Non-linearity)
    h = torch.tanh(hpreact) # 隐藏层激活
    logits = h @ W2 + b2 # 输出层
    loss = F.cross_entropy(logits, Yb) # 计算损失

    # 反向传播 (backward pass)
    for p in parameters:
      p.grad = None
    # loss.backward() # 用于正确性对比，现在我们手写，所以注释掉

    # 手动反向传播！ (manual backprop!) 
    # -----------------
    # 1. 损失函数对 logits 的直接梯度 (Cross Entropy + Softmax 简化公式)
    dlogits = F.softmax(logits, 1)
    dlogits[range(n), Yb] -= 1
    dlogits /= n
    
    # 2. 第 2 层反向传播 (Linear Layer 2)
    dh = dlogits @ W2.T
    dW2 = h.T @ dlogits
    db2 = dlogits.sum(0)
    
    # 3. Tanh 激活函数反向传播
    dhpreact = (1.0 - h**2) * dh
    
    # 4. BatchNorm 反向传播 (高度优化的合并公式)
    dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
    dbnbias = dhpreact.sum(0, keepdim=True)
    dhprebn = bngain*bnvar_inv/n * (n*dhpreact - dhpreact.sum(0) - n/(n-1)*bnraw*(dhpreact*bnraw).sum(0))
    
    # 5. 第 1 层线性层反向传播 (Linear Layer 1)
    dembcat = dhprebn @ W1.T
    dW1 = embcat.T @ dhprebn
    db1 = dhprebn.sum(0)
    
    # 6. 嵌入层反向传播 (Embedding Layer)
    demb = dembcat.view(emb.shape)
    dC = torch.zeros_like(C)
    for k in range(Xb.shape[0]):
      for j in range(Xb.shape[1]):
        ix_val = Xb[k,j]
        dC[ix_val] += demb[k,j] # 将梯度累加回对应的字符向量
        
    grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]
    # -----------------

    # 参数更新 (update)
    lr = 0.1 if i < 100000 else 0.01 # 阶梯式学习率衰减
    for p, grad in zip(parameters, grads):
      # p.data += -lr * p.grad # 旧方法：弱小狗 (cheems doge) 模式
      p.data += -lr * grad # 新方法：肌肉狗模式 (swole doge)
      
    # 追踪统计数据 (track stats)
    if i % 10000 == 0: # 隔一段时间打印一次进度
      print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())

    # 当你准备好训练整个网络时，删除或注释掉下面这两行
    # if i >= 100: 
    #   break

12297
      0/ 200000: 3.7438
  10000/ 200000: 2.1679
  20000/ 200000: 2.4514
  30000/ 200000: 2.4480
  40000/ 200000: 1.9816
  50000/ 200000: 2.4504
  60000/ 200000: 2.4139
  70000/ 200000: 1.9988
  80000/ 200000: 2.2813
  90000/ 200000: 2.1402
 100000/ 200000: 1.9222
 110000/ 200000: 2.2571
 120000/ 200000: 1.9822
 130000/ 200000: 2.4723
 140000/ 200000: 2.2299
 150000/ 200000: 2.0798
 160000/ 200000: 1.9737
 170000/ 200000: 1.9075
 180000/ 200000: 1.9475
 190000/ 200000: 1.9191


In [ ]:
# 可用于检查梯度效果。
# for p,g in zip(parameters, grads):
#   cmp(str(tuple(p.shape)), g, p)

In [16]:
# 在训练结束时校准批次归一化

with torch.no_grad():
  # 将训练集传递下去
  emb = C[Xtr]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1 + b1
  # 计算整个训练集的均值/标准差
  bnmean = hpreact.mean(0, keepdim=True)
  bnvar = hpreact.var(0, keepdim=True, unbiased=True)

In [17]:
# evaluate train and val loss

@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  hpreact = embcat @ W1 + b1
  hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
  h = torch.tanh(hpreact) # (N, n_hidden)
  logits = h @ W2 + b2 # (N, vocab_size)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.1569528579711914
val 2.185638666152954


In [18]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      # ------------
      # forward pass:
      # Embedding
      emb = C[torch.tensor([context])] # (1,block_size,d)      
      embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
      hpreact = embcat @ W1 + b1
      hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
      h = torch.tanh(hpreact) # (N, n_hidden)
      logits = h @ W2 + b2 # (N, vocab_size)
      # ------------
      # Sample
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))

carlah.
ambril.
khi.
mrix.
tahya.
kaan.
kensahnen.
deliya.
jarqui.
nellara.
chaiiv.
asleigh.
ham.
jori.
quinton.
lilea.
jadiquinte.
madiaryxie.
kaeliigsti.
edde.
